# SIM 504 — Secondary Mesh Convergence (v2)

Four independent figures — one per (element type × row):

- **Triangular Row 1:** eta · shear mean · GLE mean · EDI mean · w CV · w CV area-weighted
- **Triangular Row 2:** w mean · w std · w std area-weighted · W (total energy)
- **Quad Row 1:** same as Row 1
- **Quad Row 2:** same as Row 2

X-axis: displacement (−U2 at PERN-9999997)

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'Z001_Results_Analizer':
    ROOT = ROOT.parent
RESULTS = ROOT / 'I001_Results'

SIM      = 504
LOAD_KEY = (1, 1, 2, 2)

In [ ]:
# ── x-axis: displacement ──────────────────────────────────────────────────────
with (RESULTS / f'DATA_PICK_{SIM}_A2.pkl').open('rb') as f:
    A2 = pickle.load(f)
disp = -np.array(A2['U2']['PERN-9999997'])
print(f'Displacement: {len(disp)} steps, [{disp.min():.3f}, {disp.max():.3f}] mm')

In [ ]:
# ── load secondary mesh files ─────────────────────────────────────────────────
def load_secondary(prefix, indices):
    loaded = []
    for i in indices:
        path = RESULTS / f'DATA_PICK_{SIM}_{prefix}_{i:03d}.pkl'
        if not path.exists():
            print(f'  {path.name}: NOT FOUND — skipped')
            continue
        with path.open('rb') as f:
            d = pickle.load(f)
        n_el  = d['general_information']['n_elements']
        label = f'{prefix}_{i:03d}  ({n_el} el.)'
        loaded.append((label, d))
        print(f'  {path.name}: {n_el} elements')
    return loaded

print('Triangular:')
TRI  = load_secondary('T2', [1, 2, 3, 4, 5])
print('Quad:')
QUAD = load_secondary('Q2', [1, 2, 3, 4, 5])

In [ ]:
# ── metric definitions ────────────────────────────────────────────────────────
METRICS_ROW1 = [
    ('eta',                 None,      'eta'),
    ('shear_mean',          None,      'shear mean'),
    ('gle_mean',            None,      'GLE mean'),
    ('edi_mean',            None,      'EDI mean'),
    ('w_cv',                LOAD_KEY,  'w CV'),
    ('w_cv_area_weighted',  LOAD_KEY,  'w CV area-weighted'),
]

METRICS_ROW2 = [
    ('w_mean',              LOAD_KEY,  'w mean'),
    ('w_std',               LOAD_KEY,  'w std'),
    ('w_std_area_weighted', LOAD_KEY,  'w std area-weighted'),
    ('W',                   LOAD_KEY,  'W (total energy)'),
]

COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']


def _get(d, key, subkey):
    val = d.get(key)
    if val is None:
        return None
    if subkey is not None:
        val = val.get(subkey) if isinstance(val, dict) else None
    return np.array(val) if val is not None else None


def plot_row(datasets, metrics, title):
    """One independent figure: 1 row × len(metrics) subplots."""
    n_cols = len(metrics)
    fig, axes = plt.subplots(1, n_cols, figsize=(3.8 * n_cols, 4.5))
    fig.suptitle(title, fontsize=13)

    for ax, (key, subkey, col_label) in zip(axes, metrics):
        for i, (label, d) in enumerate(datasets):
            y = _get(d, key, subkey)
            if y is None:
                continue
            n = min(len(disp), len(y))
            ax.plot(disp[:n], y[:n],
                    color=COLORS[i % len(COLORS)],
                    linewidth=1.5,
                    label=label)
        ax.set_title(col_label, fontsize=10)
        ax.set_xlabel('Displacement [mm]', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='lower center', ncol=len(datasets),
               fontsize=8, bbox_to_anchor=(0.5, -0.08))
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Triangular — Row 1 ────────────────────────────────────────────────────────
plot_row(TRI, METRICS_ROW1, f'SIM {SIM} — Triangular secondary meshes | Row 1')

In [ ]:
# ── Triangular — Row 2 ────────────────────────────────────────────────────────
plot_row(TRI, METRICS_ROW2, f'SIM {SIM} — Triangular secondary meshes | Row 2')

In [ ]:
# ── Quad — Row 1 ──────────────────────────────────────────────────────────────
plot_row(QUAD, METRICS_ROW1, f'SIM {SIM} — Quad secondary meshes | Row 1')

In [ ]:
# ── Quad — Row 2 ──────────────────────────────────────────────────────────────
plot_row(QUAD, METRICS_ROW2, f'SIM {SIM} — Quad secondary meshes | Row 2')

In [ ]:
# ── Combined plot helper ───────────────────────────────────────────────────────
# Triangular = solid lines, Quad = dashed lines.
# Colours cycle by mesh index so T2_001 and Q2_001 share the same colour.

def plot_row_combined(tri_datasets, quad_datasets, metrics, title):
    """One independent figure with tri + quad overlaid: 1 row × len(metrics) subplots."""
    n_cols = len(metrics)
    fig, axes = plt.subplots(1, n_cols, figsize=(3.8 * n_cols, 4.5))
    fig.suptitle(title, fontsize=13)

    for ax, (key, subkey, col_label) in zip(axes, metrics):
        for i, (label, d) in enumerate(tri_datasets):
            y = _get(d, key, subkey)
            if y is None:
                continue
            n = min(len(disp), len(y))
            ax.plot(disp[:n], y[:n],
                    color=COLORS[i % len(COLORS)],
                    linestyle='-', linewidth=1.5,
                    label=f'TRI {label}')

        for i, (label, d) in enumerate(quad_datasets):
            y = _get(d, key, subkey)
            if y is None:
                continue
            n = min(len(disp), len(y))
            ax.plot(disp[:n], y[:n],
                    color=COLORS[i % len(COLORS)],
                    linestyle='--', linewidth=1.5,
                    label=f'QUAD {label}')

        ax.set_title(col_label, fontsize=10)
        ax.set_xlabel('Displacement [mm]', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='lower center',
               ncol=max(len(tri_datasets), len(quad_datasets)),
               fontsize=7, bbox_to_anchor=(0.5, -0.12))
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Combined — Row 1 (Tri + Quad) ─────────────────────────────────────────────
plot_row_combined(TRI, QUAD, METRICS_ROW1, f'SIM {SIM} — Tri + Quad secondary meshes | Row 1')

In [ ]:
# ── Combined — Row 2 (Tri + Quad) ─────────────────────────────────────────────
plot_row_combined(TRI, QUAD, METRICS_ROW2, f'SIM {SIM} — Tri + Quad secondary meshes | Row 2')